ARTI308 - Machine Learning

# Credit Card Customer Segmentation Project

In this project, you will use K-Means clustering to segment [credit card customers](https://www.kaggle.com/datasets/arjunbhasin2013/ccdata/data) based on their usage behavior. This is an unsupervised learning problem because the dataset does not contain a target label for customer groups.

You will use the `CC_GENERAL.csv` dataset.

## About the Dataset

The dataset contains customer-level credit card usage behavior. Each row represents one credit card holder, and the columns describe different behavioral variables such as balance, purchases, cash advance, payments, and tenure. The goal is to group similar customers together so that the company can understand different customer segments and design better marketing strategies.

## Import Libraries

**Import the libraries you need for data analysis, visualization, preprocessing, clustering, and evaluation.**

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

## Get the Data

**Read the `CC_GENERAL.csv` file and save it in a dataframe called `df`.**

In [ ]:
df = pd.read_csv('CC_GENERAL.csv')  

**Check the first five rows of the dataset.**

In [ ]:
df.head()

**Check the shape of the dataset.**

In [ ]:
df.shape

**Check basic information about the dataset using `info()`.**

In [ ]:
df.info()

**Check summary statistics using `describe()`.**

In [ ]:
df.describe()

## Data Cleaning

The column `CUST_ID` is an identification column. It is not useful for clustering because it does not describe customer behavior.

**Drop the `CUST_ID` column from the dataframe.**

In [ ]:
df.drop('CUST_ID', axis=1, inplace=True)

**Check the missing values in each column.**

In [ ]:
df.isnull().sum()

Some columns may contain missing values.

Hint: You can handle missing values by either:
- filling them with the mean value
- or dropping the rows that contain missing values

For this project, use mean imputation.

**Fill the missing values with the mean of each column.**

In [ ]:
df.fillna(df.mean(), inplace=True)

**Check the missing values again to make sure they were handled.**

In [ ]:
df.isnull().sum()

## Exploratory Data Analysis

Before applying clustering, it is important to understand the data.

**Create histograms for the numerical columns.**

In [ ]:
df.hist(figsize=(16, 12), bins=30)
plt.suptitle('Distribution of Numerical Features', y=1.02)
plt.tight_layout()
plt.show()

**Create a correlation heatmap to understand relationships between the features.**

In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

**Create a scatter plot between `BALANCE` and `PURCHASES`.**

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df['BALANCE'], df['PURCHASES'], alpha=0.4)
plt.xlabel('Balance')
plt.ylabel('Purchases')
plt.title('Balance vs Purchases')
plt.show()

**Create a scatter plot between `BALANCE` and `CASH_ADVANCE`.**

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df['BALANCE'], df['CASH_ADVANCE'], alpha=0.4, color='orange')
plt.xlabel('Balance')
plt.ylabel('Cash Advance')
plt.title('Balance vs Cash Advance')
plt.show()

## Feature Scaling

K-Means is a distance-based algorithm. Therefore, feature scaling is very important.

The features in this dataset have very different ranges. For example, `BALANCE`, `PURCHASES`, and `CREDIT_LIMIT` may have large values, while frequency columns are between 0 and 1.

**Use StandardScaler to scale the data. Save the scaled data in a variable called `X_scaled`.**

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df.drop('Cluster', axis=1, errors='ignore'))

X_scaled[:5]

## Choosing K Intuitively

Choosing K is one of the most difficult parts of K-Means.

Since this dataset has many features, it is not easy to visually see the clusters directly.

However, we can still compare different K values using the elbow method and silhouette score.

## Elbow Method

**Create a loop that fits K-Means models for K values from 1 to 10. Save the inertia values in a list called `inertia_values`.**

In [ ]:
inertia_values = []

K_range = range(1, 11)

for k in K_range:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertia_values.append(model.inertia_)

**Plot the elbow curve.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(K_range, inertia_values, marker='o')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Choosing K')
plt.xticks(K_range)
plt.show()

**Output Interpretation**

Look at the elbow curve and try to identify where the decrease in inertia starts to slow down.

That point can suggest a reasonable value for K.

## Silhouette Score

The silhouette score helps evaluate how well-separated the clusters are.

**Create a loop that calculates the silhouette score for K values from 2 to 10. Save the scores in a list called `silhouette_scores`.**

In [ ]:
silhouette_scores = []

K_range_sil = range(2, 11)

for k in K_range_sil:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

**Plot the silhouette scores.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(K_range_sil, silhouette_scores, marker='o', color='green')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score for Different K Values')
plt.xticks(K_range_sil)
plt.show()

**Create a table showing each K value and its silhouette score.**

In [ ]:
score_table = pd.DataFrame({
    'K': list(K_range_sil),
    'Silhouette Score': silhouette_scores
})

score_table

**Output Interpretation**

A higher silhouette score usually means better clustering.

However, do not rely only on the highest value. Also consider whether the chosen K makes sense for customer segmentation.

## Create the Final K-Means Model

**Based on the elbow curve and silhouette scores, choose a final K value. Then train a final K-Means model.**

Use `random_state=42` and `n_init=10`.

In [ ]:
final_model = KMeans(n_clusters=4, random_state=42, n_init=10)
final_model.fit(X_scaled)

**Add the final cluster labels to the original dataframe in a new column called `Cluster`.**

In [ ]:
df['Cluster'] = final_model.labels_

**Check the first five rows after adding the cluster labels.**

In [ ]:
df.head()

## Cluster Analysis

Now we need to understand what each cluster means.

**Create a summary table using `groupby()` to show the mean values of each feature for each cluster.**

In [ ]:
cluster_summary = df.groupby('Cluster').mean()
cluster_summary

**Check how many customers are in each cluster.**

In [ ]:
df['Cluster'].value_counts().sort_index()

## Visualizing the Final Clusters

Since the dataset has many features, we will use PCA to reduce the data into two components only for visualization.

This visualization does not replace the original clustering. It only helps us see the clusters in a 2D plot.

**Use PCA with 2 components and plot the clusters.**

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(9, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df['Cluster'], cmap='viridis', alpha=0.5)
plt.colorbar(scatter, label='Cluster')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.title('Customer Clusters (PCA Visualization)')
plt.show()

**Output Interpretation**

The PCA plot gives a simplified 2D view of the clusters.

If the clusters are not perfectly separated, that is normal because the original dataset has many features and the plot only shows two compressed dimensions.

## Final Questions

Answer the following questions:

1. Why is this an unsupervised learning problem?

2. Why did we remove the `CUST_ID` column?

3. Which columns had missing values?

4. How did you handle the missing values?

5. Why is scaling important before applying K-Means?

6. Which K value did you choose? Explain your answer using the elbow method and silhouette score.

7. Based on the cluster summary table, describe each customer segment in your own words.

8. Which cluster may represent high-value customers?

9. Which cluster may represent customers who rely more on cash advance?

10. How can a company use these clusters for marketing strategy?

### Answers

**1. Why is this an unsupervised learning problem?**
This is an unsupervised learning problem because the dataset does not contain a predefined target label or class column. There is no column telling us which group each customer belongs to. We let the K-Means algorithm discover hidden patterns and group similar customers together on its own, without any supervision.

**2. Why did we remove the `CUST_ID` column?**
The `CUST_ID` column is just an identifier that uniquely labels each customer. It carries no information about customer behavior or spending patterns. Including it would confuse the clustering algorithm because it would try to compute distances based on meaningless ID numbers. Removing it ensures the algorithm only uses behavioral features.

**3. Which columns had missing values?**
Two columns had missing values: `CREDIT_LIMIT` (1 missing value) and `MINIMUM_PAYMENTS` (313 missing values).

**4. How did you handle the missing values?**
We used mean imputation — each missing value was replaced with the mean of its respective column. This approach preserves all rows and avoids reducing the dataset size, which is especially important for `MINIMUM_PAYMENTS` since it had 313 missing entries.

**5. Why is scaling important before applying K-Means?**
K-Means calculates distances between data points. If features have very different ranges (for example, `BALANCE` can be in the thousands while `PURCHASES_FREQUENCY` is between 0 and 1), the features with larger values will dominate the distance calculations and bias the clustering. StandardScaler transforms all features to have a mean of 0 and a standard deviation of 1, so each feature contributes equally to the clustering.

**6. Which K value did you choose? Explain your answer using the elbow method and silhouette score.**
We chose **K = 4**. The elbow curve shows a clear drop in inertia from K=1 to K=4, after which the improvement starts to slow down — suggesting the elbow is around K=4. The silhouette score peaks at K=3 (≈0.250) but drops at K=4 (≈0.198). We chose K=4 as a balance between the two methods: it still has a reasonable silhouette score and produces four distinct, business-interpretable customer segments rather than just three broader groups.

**7. Based on the cluster summary table, describe each customer segment in your own words.**
- **Cluster 0 — Moderate Balanced Customers:** These customers have moderate balances (~$895), make a reasonable number of purchases (~$1,236), and carry a low cash advance (~$211). They represent an average, well-rounded customer segment.
- **Cluster 1 — High-Value Active Shoppers:** This group has the highest purchases (~$7,682), the highest credit limit (~$9,697), and the highest payments (~$7,289). They are active, big spenders who pay off much of their balance.
- **Cluster 2 — Cash Advance Dependent Customers:** These customers have the highest cash advance usage (~$4,521) and the highest balances (~$4,602) but very low purchases (~$502). They rely on their credit cards mostly for cash withdrawals rather than purchases.
- **Cluster 3 — Low-Activity Customers:** This is the largest group, with the lowest purchases (~$270), low credit limits (~$3,278), and low payments (~$975). These customers use their credit card minimally.

**8. Which cluster may represent high-value customers?**
**Cluster 1** represents high-value customers. They have the highest purchases, highest credit limit, and the highest payments, meaning they spend a lot and also pay their balances, making them valuable and financially reliable customers.

**9. Which cluster may represent customers who rely more on cash advance?**
**Cluster 2** represents customers who rely heavily on cash advances. Their average cash advance amount (~$4,521) is more than eight times higher than the other clusters, and they have a high cash advance frequency. These customers treat their credit card more like a short-term loan than a shopping tool.

**10. How can a company use these clusters for marketing strategy?**
- **Cluster 0 (Moderate customers):** Offer rewards programs or cashback incentives to increase their engagement and nudge them toward becoming high-value shoppers.
- **Cluster 1 (High-value shoppers):** Provide exclusive loyalty rewards, travel benefits, or premium card upgrades to retain these valuable customers.
- **Cluster 2 (Cash advance users):** Offer financial planning resources, lower cash advance fees, or personal loan products that might better meet their needs and reduce financial risk.
- **Cluster 3 (Low-activity customers):** Send targeted promotions, limited-time offers, or sign-up bonuses to encourage more card usage before they churn.
